# 🧪 tokenizer / template / parser

qwen3_xml と qwen3 reasoning parser が実出力を捌けるか。


In [ ]:
import subprocess
import sys

print("=== pip install vllm==0.32.0 ===")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input",
                    "vllm==0.32.0"], capture_output=True, text=True, timeout=5400)
tail = ((r.stdout or "") + (r.stderr or "")).strip().splitlines()
print("   rc:", r.returncode, "|", " / ".join(tail[-3:])[:300])
r = subprocess.run([sys.executable, "-c",
                    "import vllm, torch, transformers;"
                    "print(vllm.__version__, torch.__version__, transformers.__version__)"],
                   capture_output=True, text=True)
print("   versions:", (r.stdout or r.stderr).strip()[:300])
def find_root():
    """dataset のマウント位置は 2 通りあるので config.json を探す。"""
    import glob
    slug = "qwen38-27b-uncensored-hauhaucs-awq-mtp"
    ordered = []
    for pat in ("/kaggle/input/*/config.json", "/kaggle/input/*/*/config.json",
                "/kaggle/input/*/*/*/config.json"):
        ordered += glob.glob(pat)
    cands = ([c for c in ordered if slug in c]
             or [c for c in ordered if "qwen" in c.lower()]
             or ordered)
    print("config.json 候補:", cands[:3])
    if not cands:
        print("/kaggle/input 直下:", glob.glob("/kaggle/input/*")[:10])
        raise SystemExit("dataset が見つからない")
    return cands[0].rsplit("/", 1)[0]

ROOT = find_root()

CASE = r"""
import json
import sys

ROOT = sys.argv[1]

print("=== tokenizer ===")
from vllm.tokenizers import get_tokenizer
tok = get_tokenizer(ROOT, trust_remote_code=True)
print("  type:", type(tok).__name__, "| vocab:", getattr(tok, "vocab_size", None))

tpl = tok.get_chat_template(None, tools=None)
print("  template len:", len(tpl) if tpl else None)
print("  has <think>:", "<think>" in (tpl or ""))
print("  has <tool_call>:", "<tool_call>" in (tpl or ""))

MSGS = [{"role": "user", "content": "東京の天気は？"}]
TOOLS = [{"type": "function", "function": {
    "name": "get_weather",
    "description": "天気を返す",
    "parameters": {"type": "object",
                   "properties": {"city": {"type": "string"}},
                   "required": ["city"]}}}]

print("\n=== render（tools なし）===")
s = tok.apply_chat_template(MSGS, tokenize=False, add_generation_prompt=True)
print(repr(s[-160:]))
print("  ids:", len(tok.encode(s)))

print("\n=== render（tools あり）===")
s2 = tok.apply_chat_template(MSGS, tools=TOOLS, tokenize=False,
                             add_generation_prompt=True)
print("  len:", len(s2), "ids:", len(tok.encode(s2)))
print("  tools 差分:", len(s2) - len(s))
i = s2.find("<tools>")
print("  <tools> 出現:", i)
print(repr(s2[i:i + 320]))
print("  末尾:", repr(s2[-160:]))

print("\n=== reasoning_effort ===")
for eff in ("low", "medium", "xhigh"):
    try:
        t = tok.apply_chat_template(
            MSGS, tokenize=False, add_generation_prompt=True,
            chat_template_kwargs={"reasoning_effort": eff})
        print(f"  {eff:7s} ids={len(tok.encode(t))}")
    except Exception as e:
        print(f"  {eff:7s} {type(e).__name__}: {str(e)[:160]}")

print("\n=== tool parser（qwen3_xml）===")
from vllm.tool_parsers import ToolParserManager
cls = ToolParserManager.get_tool_parser("qwen3_xml")
print("  class:", cls)
print("  module:", cls.__module__)
tp = cls(tok, tools=TOOLS)
print("  instance OK")

SAMPLE = (
    "<think>\nweather を調べる\n</think>\n"
    "<tool_call>\n<function=get_weather>\n"
    "<parameter=city>東京</parameter>\n</function>\n</tool_call>"
)
try:
    r = tp.extract_tool_calls(SAMPLE, request=None)
    print("  tools_called:", r.tools_called)
    print("  tool_calls:", json.dumps(r.tool_calls, ensure_ascii=False)[:400])
    print("  content:", repr((r.content or "")[:120]))
except TypeError as e:
    print("  signature 違い:", e)
    r = tp.extract_tool_calls(SAMPLE)
    print("  tool_calls:", json.dumps(r.tool_calls, ensure_ascii=False)[:400])
except Exception as e:
    import traceback
    traceback.print_exc()

print("\n=== reasoning parser（qwen3）===")
from vllm.reasoning import ReasoningParserManager
rcls = None
for mgr, name in ((ReasoningParserManager, "qwen3"),):
    try:
        rcls = mgr.get_reasoning_parser(name)
        print("  class:", rcls, "| module:", rcls.__module__)
    except Exception as e:
        print("  [NG]", name, type(e).__name__, str(e)[:200])
if rcls:
    for ctor in (lambda: rcls(tok), lambda: rcls(tokenizer=tok)):
        try:
            rp = ctor()
            print("  instance OK")
            break
        except Exception as e:
            print("  ctor NG:", type(e).__name__, str(e)[:160])
    else:
        rp = None
    if rp is not None:
        txt = "<think>\n考え中\n</think>\n\nこんにちは"
        try:
            rc, content = rp.extract_reasoning(txt, request=None)
        except TypeError:
            rc, content = rp.extract_reasoning(txt)
        print("  reasoning:", repr(rc))
        print("  content  :", repr(content))

print("\n=== 終了 ===")
"""

import os
import subprocess
import sys
import tempfile

case = os.path.join(tempfile.gettempdir(), "parsercase.py")
open(case, "w").write(CASE)
r = subprocess.run([sys.executable, case, ROOT],
                   capture_output=True, text=True, timeout=3600)
print((r.stdout or "")[-16000:])
if r.returncode != 0:
    print("=== stderr ===")
    print((r.stderr or "")[-4000:])
